##  4. Классификационные модели

Теперь переходим к задачам классификации. У нас уже есть бинарные столбцы:
- IC50_high (1 если IC50 > медианы)
- CC50_high
- SI_high_median
- SI_high_8

Для каждой из этих целевых переменных обучим три модели:
1. LogisticRegression (базовый уровень)
2. RandomForestClassifier (ансамбль)
3. XGBClassifier (бустинг)

Оценим качество по accuracy, F1 и AUC-ROC. Используем тот же сплит train/test (75/25) с фиксацией random_state=42 и стратификацией.

Теперь переходим к задачам классификации. У нас уже есть бинарные столбцы:
- IC50_high (1 если IC50 > медианы)
- CC50_high
- SI_high_median
- SI_high_8

Для каждой из этих целевых переменных обучим три модели:
1. LogisticRegression (базовый уровень)
2. RandomForestClassifier (ансамбль)
3. XGBClassifier (бустинг)

Оценим качество по accuracy, F1 и AUC-ROC. Используем тот же сплит train/test (75/25) с фиксацией random_state=42 и стратификацией.

Классификация: превышает ли значение IC50 медианное значение выборки

In [8]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

X_train = pd.read_csv('/content/X_train.csv')
X_test = pd.read_csv('/content/X_test.csv')

# Функция для обучения и оценки
def evaluate_classifier(y_train_path, y_test_path, model, model_name, task_name):
    y_train = pd.read_csv(y_train_path).values.ravel()
    y_test = pd.read_csv(y_test_path).values.ravel()

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"\n{task_name} - {model_name}")
    print(f"  Accuracy: {accuracy_score(y_test, y_pred):.3f}")
    print(f"  F1-score: {f1_score(y_test, y_pred):.3f}")
    print(f"  ROC-AUC:  {roc_auc_score(y_test, y_proba):.3f}")

print("="*60)
print("КЛАССИФИКАЦИЯ IC50 (выше/ниже медианы)")
print("="*60)

# Logistic Regression
evaluate_classifier(
    'y_train_ic50_high.csv', 'y_test_ic50_high.csv',
    LogisticRegression(max_iter=1000, random_state=42),
    "LogisticRegression", "IC50"
)

# Random Forest
evaluate_classifier(
    'y_train_ic50_high.csv', 'y_test_ic50_high.csv',
    RandomForestClassifier(n_estimators=100, random_state=42),
    "RandomForest", "IC50"
)

# XGBoost
evaluate_classifier(
    'y_train_ic50_high.csv', 'y_test_ic50_high.csv',
    XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
    "XGBoost", "IC50"
)

КЛАССИФИКАЦИЯ IC50 (выше/ниже медианы)

IC50 - LogisticRegression
  Accuracy: 0.479
  F1-score: 0.648
  ROC-AUC:  0.481

IC50 - RandomForest
  Accuracy: 0.773
  F1-score: 0.756
  ROC-AUC:  0.826

IC50 - XGBoost
  Accuracy: 0.760
  F1-score: 0.754
  ROC-AUC:  0.802


Вывод:в ходе выполнения классификации IC50 установлено, что ансамблевые методы (Random Forest и XGBoost) значительно превосходят линейную логистическую регрессию. Random Forest показал наивысшее качество (AUC = 82,6%, Accuracy = 77,3%, F1-score = 75,6%(почти такой же показатель и у XGBoost )), что подтверждает наличие нелинейных зависимостей между молекулярными признаками и ингибирующей активностью. Полученная модель пригодна для практического использования в задачах виртуального скрининга.

